In [9]:
import pandas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import xgboost as xgb
from xgboost.callback import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

In [10]:
data = pandas.read_csv("E:\Battey_Gauge\data\Merged\\battery_training_data.csv")

In [11]:
data.head()

,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,Time to Depletion
0,3.44,12.35,0.057333,0.057333,42.4840,53.942667,56451.627907
1,6.88,12.22,0.114667,0.172000,84.0736,53.828000,28165.813953
2,6.88,12.16,0.114667,0.286667,83.6608,53.713333,28105.813953
3,6.88,12.17,0.114667,0.401333,83.7296,53.598667,28045.813953
4,6.87,12.18,0.114500,0.515833,83.6766,53.484167,28026.637555


In [12]:
data.shape

(5437, 7)

In [13]:
TARGET_VARIABLE = 'Time to Depletion'

In [14]:
Y = data[TARGET_VARIABLE]
X = data.drop( 'Ah Out', axis=1)
X = X.drop( 'Cumulative Actual Disch Ah', axis=1)
X = X.drop( 'Power', axis=1)
X = X.drop( 'Remaining Capacity', axis=1)
X = X.drop(TARGET_VARIABLE, axis=1)

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

print("Numerical Featrures are : ", numerical_features)

Numerical Featrures are :  Index(['Current', 'Voltage'], dtype='object')


In [15]:
numerical_transformer = Pipeline(steps=[('pass','passthrough')])

In [16]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features)
]
    ,remainder='passthrough')

In [17]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.1, random_state=42)

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [19]:
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mae'
)

In [20]:
early_stopping = EarlyStopping(
    rounds=10,
    metric_name='mae',
    save_best=True,
)

In [21]:
eval_set = [(X_test_processed, Y_test)]
xgb_model.set_params(callbacks=[early_stopping])
xgb_model.fit(X_train_processed, Y_train,
              eval_set=eval_set,
              verbose=False)

XGBRegressor(base_score=None, booster=None,
             callbacks=[<xgboost.callback.EarlyStopping object at 0x000001EE3224B610>],
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='mae', feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [22]:
Y_pred = xgb_model.predict(X_test_processed)

In [23]:
mae = mean_absolute_error(Y_test, Y_pred)

print(f"Mean Absolute Error is : {mae:.2f}")

Mean Absolute Error is : 7786.44
